# 集成学习第六课：XGBoost

## 本课目标

XGBoost 可以先理解成：

GBDT 的工程增强版和数学增强版。

它仍然是一轮一轮加树：

$$
F_t(x)
=
F_{t-1}(x)
+
\nu f_t(x)
$$

但它比普通 GBDT 多考虑了几件事：

- 用二阶导数帮助判断怎么修正。
- 在目标函数里加入正则化，控制树不要太复杂。
- 用切分增益判断一个切分值不值得切。

本课先学 XGBoost 回归任务里的核心数学。

## 1. XGBoost 和 GBDT 的关系

GBDT 的核心是：

每一轮新训练一棵树，修正旧模型还没预测好的部分。

XGBoost 也是这个思路。

但是 XGBoost 不满足于只说一句：

新树去拟合残差。

它会先问一个更底层的问题：

这一轮到底应该加一棵什么树，才算“最好”？

要回答这个问题，就必须先定义“好”的标准。

XGBoost 的标准分成两部分。

第一部分：预测要准。

也就是模型预测值 $\hat{y}$ 要接近真实值 $y$。

第二部分：树不要太复杂。

也就是不要为了训练集上的一点点误差，长出很多叶子，或者让某些叶子的输出特别大。

所以 XGBoost 的推导不是凭空变出一个公式。

它是在把这句话数学化：

我们要找一棵新树，让“训练误差”和“树的复杂度”加起来最小。

## 2. XGBoost 的目标函数从哪里来？

先不要看公式，先看问题。

训练集里有 $m$ 个样本：

$$
(x_1,y_1),(x_2,y_2),\ldots,(x_m,y_m)
$$

对第 $i$ 个样本，模型会给出一个预测值：

$$
\hat{y}_i
$$

如果预测值和真实值差得很远，损失就大。

用损失函数表示就是：

$$
L(y_i,\hat{y}_i)
$$

这只是一个样本的损失。

整个训练集有 $m$ 个样本，所以总训练损失就是每个样本损失加起来：

$$
\sum_{i=1}^{m}
L(y_i,\hat{y}_i)
$$

这一步没有玄学。

它只是说：

模型不能只照顾一个样本，而要照顾全部训练样本。

现在进入 Boosting。

Boosting 的模型不是一次训练完，而是一轮一轮往里面加新模型。

第 $t-1$ 轮结束后，旧模型是：

$$
F_{t-1}(x)
$$

它对第 $i$ 个样本的旧预测是：

$$
\hat{y}_i^{(t-1)}
=
F_{t-1}(x_i)
$$

第 $t$ 轮新加一棵树：

$$
f_t(x)
$$

这棵树对第 $i$ 个样本的输出是：

$$
f_t(x_i)
$$

Boosting 是加法模型。

也就是说，新模型等于旧模型加上新树：

$$
F_t(x)
=
F_{t-1}(x)
+
f_t(x)
$$

把 $x_i$ 代进去：

$$
F_t(x_i)
=
F_{t-1}(x_i)
+
f_t(x_i)
$$

因为：

$$
\hat{y}_i^{(t)}
=
F_t(x_i)
$$

并且：

$$
\hat{y}_i^{(t-1)}
=
F_{t-1}(x_i)
$$

所以第 $t$ 轮更新后的预测是：

$$
\hat{y}_i^{(t)}
=
\hat{y}_i^{(t-1)}
+
f_t(x_i)
$$

现在把这件事放回“训练集总损失”。

第 $t$ 轮之后，第 $i$ 个样本的损失是：

$$
L(y_i,\hat{y}_i^{(t)})
$$

全部样本的损失是：

$$
\sum_{i=1}^{m}
L(y_i,\hat{y}_i^{(t)})
$$

如果只最小化这个式子，树可能会变得很复杂。

所以 XGBoost 还要给树加复杂度惩罚。

第 $k$ 棵树的复杂度写成：

$$
\Omega(f_k)
$$

如果模型一共有 $t$ 棵树，那么完整目标可以写成：

$$
Obj_{\mathrm{all}}^{(t)}
=
\sum_{i=1}^{m}
L(y_i,\hat{y}_i^{(t)})
+
\sum_{k=1}^{t}
\Omega(f_k)
$$

现在注意第 $t$ 轮我们到底在优化什么。

前 $t-1$ 棵树已经训练好了：

$$
f_1(x),f_2(x),\ldots,f_{t-1}(x)
$$

它们在这一轮是固定的，不会再变。

所以：

$$
\sum_{k=1}^{t}
\Omega(f_k)
=
\sum_{k=1}^{t-1}
\Omega(f_k)
+
\Omega(f_t)
$$

其中：

$$
\sum_{k=1}^{t-1}
\Omega(f_k)
$$

只和旧树有关，和这轮要找的新树 $f_t(x)$ 无关。

第 $t$ 轮比较不同新树时，这一项对每个候选新树都一样。

所以它只是一个常数。

比如比较两个数：

$$
A+C
$$

和：

$$
B+C
$$

如果 $C$ 一样，那么谁小只取决于 $A$ 和 $B$。

所以第 $t$ 轮可以只写和新树有关的目标：

$$
Obj^{(t)}
=
\sum_{i=1}^{m}
L(y_i,\hat{y}_i^{(t)})
+
\Omega(f_t)
$$

再把第 $t$ 轮预测：

$$
\hat{y}_i^{(t)}
=
\hat{y}_i^{(t-1)}
+
f_t(x_i)
$$

代进去：

$$
Obj^{(t)}
=
\sum_{i=1}^{m}
L\left(
y_i,
\hat{y}_i^{(t-1)}+f_t(x_i)
\right)
+
\Omega(f_t)
$$

这就是截图里那个公式真正的来源。

说人话：

第 $t$ 轮旧模型已经固定了。

我们只是在问：

现在加哪一棵树 $f_t(x)$，能让所有样本的损失下降，同时这棵树本身别太复杂？

## 3. 为什么要用泰勒展开？

现在目标函数里最麻烦的是这一项：

$$
L\left(
y_i,
\hat{y}_i^{(t-1)}+f_t(x_i)
\right)
$$

原因是：

我们还不知道新树 $f_t(x)$ 长什么样。

如果直接在原始损失函数上搜索所有可能的树，问题会很难。

XGBoost 的想法是：

旧预测 $\hat{y}_i^{(t-1)}$ 已经有了。

新树 $f_t(x_i)$ 可以看作在旧预测旁边迈出的一小步。

先把损失函数看成一个只关于预测值的函数。

对第 $i$ 个样本，真实值 $y_i$ 是固定的。

令：

$$
\phi_i(z)
=
L(y_i,z)
$$

这里的 $z$ 表示模型预测值。

旧预测是：

$$
z_0
=
\hat{y}_i^{(t-1)}
$$

新树带来的变化是：

$$
\Delta
=
f_t(x_i)
$$

新预测就是：

$$
z_0+\Delta
$$

所以这个样本的新损失可以写成：

$$
L\left(
y_i,
\hat{y}_i^{(t-1)}+f_t(x_i)
\right)
=
\phi_i(z_0+\Delta)
$$

二阶泰勒展开告诉我们：

一个函数在 $z_0$ 附近，可以用一个二次函数近似：

$$
\phi_i(z_0+\Delta)
\approx
\phi_i(z_0)
+
\phi_i'(z_0)\Delta
+
\frac{1}{2}\phi_i''(z_0)\Delta^2
$$

现在把 $z_0$ 和 $\Delta$ 换回 XGBoost 里的东西。

先看第一项：

$$
\phi_i(z_0)
=
L(y_i,\hat{y}_i^{(t-1)})
$$

再看第二项。

$\phi_i'(z_0)$ 是损失函数对预测值的一阶导数：

$$
\phi_i'(z_0)
=
\left.
\frac{\partial L(y_i,\hat{y}_i)}
{\partial \hat{y}_i}
\right|_{\hat{y}_i=\hat{y}_i^{(t-1)}}
$$

XGBoost 把它记作：

$$
g_i
$$

又因为：

$$
\Delta
=
f_t(x_i)
$$

所以第二项变成：

$$
g_i f_t(x_i)
$$

再看第三项。

$\phi_i''(z_0)$ 是损失函数对预测值的二阶导数：

$$
\phi_i''(z_0)
=
\left.
\frac{\partial^2 L(y_i,\hat{y}_i)}
{\partial \hat{y}_i^2}
\right|_{\hat{y}_i=\hat{y}_i^{(t-1)}}
$$

XGBoost 把它记作：

$$
h_i
$$

而：

$$
\Delta^2
=
f_t(x_i)^2
$$

所以第三项变成：

$$
\frac{1}{2}h_i f_t(x_i)^2
$$

把三项合起来：

$$
L\left(
y_i,
\hat{y}_i^{(t-1)}+f_t(x_i)
\right)
\approx
L(y_i,\hat{y}_i^{(t-1)})
+
g_i f_t(x_i)
+
\frac{1}{2}h_i f_t(x_i)^2
$$

说人话：

XGBoost 不是凭空写出 $g_i$ 和 $h_i$。

它是把“新树加入后损失会怎么变”用二次函数近似。

$g_i$ 来自一阶导数，表示下降方向。

$h_i$ 来自二阶导数，表示这一步该走得多谨慎。

## 4. 去掉和新树无关的常数项

上一节得到单个样本的近似：

$$
L\left(
y_i,
\hat{y}_i^{(t-1)}+f_t(x_i)
\right)
\approx
L(y_i,\hat{y}_i^{(t-1)})
+
g_i f_t(x_i)
+
\frac{1}{2}h_i f_t(x_i)^2
$$

把它放回全部样本的目标函数：

$$
Obj^{(t)}
\approx
\sum_{i=1}^{m}
\left[
L(y_i,\hat{y}_i^{(t-1)})
+
g_i f_t(x_i)
+
\frac{1}{2}h_i f_t(x_i)^2
\right]
+
\Omega(f_t)
$$

把求和拆开：

$$
Obj^{(t)}
\approx
\sum_{i=1}^{m}
L(y_i,\hat{y}_i^{(t-1)})
+
\sum_{i=1}^{m}
g_i f_t(x_i)
+
\sum_{i=1}^{m}
\frac{1}{2}h_i f_t(x_i)^2
+
\Omega(f_t)
$$

第一项：

$$
\sum_{i=1}^{m}
L(y_i,\hat{y}_i^{(t-1)})
$$

只和旧预测有关。

旧预测 $\hat{y}_i^{(t-1)}$ 在第 $t$ 轮已经固定。

所以这一项不管你选哪棵新树都一样。

它就是一个常数。

如果两个候选新树分别对应：

$$
C+A
$$

和：

$$
C+B
$$

这里的 $C$ 一样，那么比较大小只需要比较：

$$
A
$$

和：

$$
B
$$

所以第 $t$ 轮真正需要优化的是：

$$
\widetilde{Obj}^{(t)}
=
\sum_{i=1}^{m}
g_i f_t(x_i)
+
\sum_{i=1}^{m}
\frac{1}{2}h_i f_t(x_i)^2
+
\Omega(f_t)
$$

通常也写成：

$$
\widetilde{Obj}^{(t)}
=
\sum_{i=1}^{m}
\left[
g_i f_t(x_i)
+
\frac{1}{2}h_i f_t(x_i)^2
\right]
+
\Omega(f_t)
$$

说人话：

旧模型已经造成的损失，不能靠“选择第 $t$ 棵树的形状”直接改变。

这一轮真正能改变的是：

新树 $f_t(x)$ 让损失增加多少或者减少多少。

## 5. 树的正则化项是什么意思？

现在看：

$$
\Omega(f)
$$

它不是从数据损失里推出来的。

它是人为加进去的约束，用来表达：

树越复杂，代价越大。

XGBoost 里一棵树可以这样理解：

每个样本先被分到某个叶子。

然后这个叶子给出一个固定输出值。

如果一棵树有 $T$ 个叶子，叶子输出分别是：

$$
w_1,w_2,\ldots,w_T
$$

那么 XGBoost 把复杂度写成两部分：

$$
\Omega(f)
=
\gamma T
+
\frac{1}{2}\lambda
\sum_{j=1}^{T}
w_j^2
$$

第一部分：

$$
\gamma T
$$

表示叶子数量的惩罚。

如果多长一个叶子，目标函数就多付出：

$$
\gamma
$$

所以如果一个切分带来的收益还不如 $\gamma$，那就不值得切。

第二部分：

$$
\frac{1}{2}\lambda
\sum_{j=1}^{T}
w_j^2
$$

表示叶子输出值的惩罚。

如果某个叶子输出 $w_j$ 特别大，说明这棵树对预测做了很猛的修正。

XGBoost 会惩罚这种很猛的修正，避免模型太激进。

为什么前面有：

$$
\frac{1}{2}
$$

主要是为了后面求导更干净。

因为：

$$
\frac{d}{dw_j}
\left(
\frac{1}{2}\lambda w_j^2
\right)
=
\lambda w_j
$$

如果没有这个 $\frac{1}{2}$，后面公式会多一个 $2$，不影响思想，只是不够简洁。

说人话：

$\gamma$ 管“叶子数量”。

$\lambda$ 管“叶子输出幅度”。

一个管树长多大，一个管每个叶子说话多重。

## 6. 按叶子把目标函数合并

现在从第 $t$ 轮真正要优化的目标开始：

$$
\widetilde{Obj}^{(t)}
=
\sum_{i=1}^{m}
\left[
g_i f_t(x_i)
+
\frac{1}{2}h_i f_t(x_i)^2
\right]
+
\Omega(f_t)
$$

把正则化项展开：

$$
\Omega(f_t)
=
\gamma T
+
\frac{1}{2}\lambda
\sum_{j=1}^{T}
w_j^2
$$

代进去：

$$
\widetilde{Obj}^{(t)}
=
\sum_{i=1}^{m}
\left[
g_i f_t(x_i)
+
\frac{1}{2}h_i f_t(x_i)^2
\right]
+
\gamma T
+
\frac{1}{2}\lambda
\sum_{j=1}^{T}
w_j^2
$$

现在利用树的结构。

一棵树会把样本分到叶子里。

第 $j$ 个叶子里的样本集合记为：

$$
R_j
$$

如果第 $i$ 个样本落在第 $j$ 个叶子里，那么这棵树对它的输出就是该叶子的输出：

$$
f_t(x_i)=w_j
$$

所以所有样本的求和，可以改成按叶子求和。

原来的样本求和：

$$
\sum_{i=1}^{m}
\left[
g_i f_t(x_i)
+
\frac{1}{2}h_i f_t(x_i)^2
\right]
$$

可以拆成：

$$
\sum_{j=1}^{T}
\sum_{i\in R_j}
\left[
g_i w_j
+
\frac{1}{2}h_i w_j^2
\right]
$$

因为在同一个叶子里，$w_j$ 对这个叶子里的所有样本都一样。

所以第 $j$ 个叶子的那部分是：

$$
\sum_{i\in R_j}
\left[
g_i w_j
+
\frac{1}{2}h_i w_j^2
\right]
$$

把它拆开：

$$
\sum_{i\in R_j}
g_i w_j
+
\sum_{i\in R_j}
\frac{1}{2}h_i w_j^2
$$

由于 $w_j$ 在这个叶子里是常数，可以提出去：

$$
w_j
\sum_{i\in R_j}
g_i
+
\frac{1}{2}w_j^2
\sum_{i\in R_j}
h_i
$$

为了写起来不拖沓，定义：

$$
G_j
=
\sum_{i\in R_j}
g_i
$$

$$
H_j
=
\sum_{i\in R_j}
h_i
$$

于是第 $j$ 个叶子的样本损失近似项变成：

$$
G_jw_j
+
\frac{1}{2}H_jw_j^2
$$

这个叶子自己的输出惩罚是：

$$
\frac{1}{2}\lambda w_j^2
$$

所以第 $j$ 个叶子的目标函数是：

$$
Obj_j(w_j)
=
G_jw_j
+
\frac{1}{2}H_jw_j^2
+
\frac{1}{2}\lambda w_j^2
$$

后两项可以合并：

$$
\frac{1}{2}H_jw_j^2
+
\frac{1}{2}\lambda w_j^2
=
\frac{1}{2}
\left(H_j+\lambda\right)
w_j^2
$$

所以：

$$
Obj_j(w_j)
=
G_jw_j
+
\frac{1}{2}
\left(H_j+\lambda\right)
w_j^2
$$

说人话：

一棵树的每个叶子只输出一个数。

因此这个叶子到底输出多少，不需要逐个样本单独决定。

只需要看这个叶子里所有样本的梯度合计 $G_j$，以及二阶梯度合计 $H_j$。

## 7. 叶子最优输出怎么推出来？

现在只看一个叶子：

$$
Obj_j(w_j)
=
G_jw_j
+
\frac{1}{2}
\left(H_j+\lambda\right)
w_j^2
$$

这是一个关于 $w_j$ 的二次函数。

我们要找让它最小的 $w_j$。

对 $w_j$ 求导：

$$
\frac{dObj_j}{dw_j}
=
G_j
+
\left(H_j+\lambda\right)w_j
$$

令导数等于 $0$：

$$
G_j
+
\left(H_j+\lambda\right)w_j
=
0
$$

移项：

$$
\left(H_j+\lambda\right)w_j
=
-G_j
$$

两边除以：

$$
H_j+\lambda
$$

得到：

$$
w_j^*
=
-
\frac{G_j}{H_j+\lambda}
$$

这就是 XGBoost 叶子输出值的公式。

说人话：

- 如果这个叶子里的梯度和 $G_j$ 很大，说明当前预测整体偏离方向明显，叶子输出就会更大。
- 如果 $H_j+\lambda$ 很大，说明分母大，叶子输出会被压小。
- $\lambda$ 越大，叶子输出越保守。

## 8. 把最优叶子值代回去

刚才得到：

$$
w_j^*
=
-
\frac{G_j}{H_j+\lambda}
$$

现在把它代回叶子目标：

$$
Obj_j(w_j)
=
G_jw_j
+
\frac{1}{2}
\left(H_j+\lambda\right)
w_j^2
$$

第一项：

$$
G_jw_j^*
=
G_j
\left(
-
\frac{G_j}{H_j+\lambda}
\right)
=
-
\frac{G_j^2}{H_j+\lambda}
$$

第二项：

$$
\frac{1}{2}
\left(H_j+\lambda\right)
\left(w_j^*\right)^2
$$

因为：

$$
\left(w_j^*\right)^2
=
\frac{G_j^2}{\left(H_j+\lambda\right)^2}
$$

所以：

$$
\frac{1}{2}
\left(H_j+\lambda\right)
\left(w_j^*\right)^2
=
\frac{1}{2}
\frac{G_j^2}{H_j+\lambda}
$$

两项相加：

$$
-
\frac{G_j^2}{H_j+\lambda}
+
\frac{1}{2}
\frac{G_j^2}{H_j+\lambda}
=
-
\frac{1}{2}
\frac{G_j^2}{H_j+\lambda}
$$

所以一个叶子在最优输出下的目标函数值是：

$$
Obj_j^*
=
-
\frac{1}{2}
\frac{G_j^2}{H_j+\lambda}
$$

整棵树有 $T$ 个叶子，再加上叶子数量惩罚：

$$
Obj^*
=
-
\frac{1}{2}
\sum_{j=1}^{T}
\frac{G_j^2}{H_j+\lambda}
+
\gamma T
$$

这个式子非常重要，因为后面判断切分好不好就靠它。

## 9. 切分增益怎么来的？

前面已经知道，一个叶子在最优输出下的目标函数值是：

$$
Obj_j^*
=
-
\frac{1}{2}
\frac{G_j^2}{H_j+\lambda}
$$

整棵树还要加叶子数量惩罚：

$$
\gamma T
$$

现在考虑一个节点到底要不要切开。

切分前，这个节点自己就是一个叶子。

它里面所有样本的一阶梯度和是：

$$
G
$$

它里面所有样本的二阶梯度和是：

$$
H
$$

切分前的最优目标值是：

$$
Obj_{\mathrm{parent}}^*
=
-
\frac{1}{2}
\frac{G^2}{H+\lambda}
+
\gamma
$$

这里的 $\gamma$ 表示：

这个父节点作为一个叶子，要付出一个叶子的复杂度代价。

现在假设把它切成左右两个叶子。

左叶子的一阶梯度和、二阶梯度和是：

$$
G_L,\quad H_L
$$

右叶子的一阶梯度和、二阶梯度和是：

$$
G_R,\quad H_R
$$

切分后，左叶子的最优目标值是：

$$
-
\frac{1}{2}
\frac{G_L^2}{H_L+\lambda}
$$

右叶子的最优目标值是：

$$
-
\frac{1}{2}
\frac{G_R^2}{H_R+\lambda}
$$

两个叶子一共要付出两个叶子的复杂度代价：

$$
2\gamma
$$

所以切分后的目标值是：

$$
Obj_{\mathrm{split}}^*
=
-
\frac{1}{2}
\frac{G_L^2}{H_L+\lambda}
-
\frac{1}{2}
\frac{G_R^2}{H_R+\lambda}
+
2\gamma
$$

也可以写成：

$$
Obj_{\mathrm{split}}^*
=
-
\frac{1}{2}
\left[
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
\right]
+
2\gamma
$$

切分好不好，看的是：

切分后目标函数有没有变小。

所以定义切分增益：

$$
Gain
=
Obj_{\mathrm{parent}}^*
-
Obj_{\mathrm{split}}^*
$$

把两个式子代进去：

$$
Gain
=
\left[
-
\frac{1}{2}
\frac{G^2}{H+\lambda}
+
\gamma
\right]
-
\left[
-
\frac{1}{2}
\left(
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
\right)
+
2\gamma
\right]
$$

去括号：

$$
Gain
=
-
\frac{1}{2}
\frac{G^2}{H+\lambda}
+
\gamma
+
\frac{1}{2}
\left(
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
\right)
-
2\gamma
$$

把同类项放到一起：

$$
Gain
=
\frac{1}{2}
\left[
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
-
\frac{G^2}{H+\lambda}
\right]
-
\gamma
$$

这就是 XGBoost 的切分增益公式。

说人话：

左边和右边分开以后，能让目标函数下降多少？

如果下降的收益，大于多长一个叶子的代价 $\gamma$，这个切分就值得。

如果：

$$
Gain>0
$$

就切。

如果：

$$
Gain\le 0
$$

就不切。

## 10. 和 GBDT 手算例子对齐

继续使用 GBDT 那节的一维回归数据：

$$
x=[1,\ 2,\ 3,\ 4,\ 5,\ 6]
$$

$$
y=[2,\ 4,\ 6,\ 8,\ 10,\ 12]
$$

初始预测取真实值均值：

$$
\hat{y}^{(0)}
=
[7,\ 7,\ 7,\ 7,\ 7,\ 7]
$$

所以第一轮残差是：

$$
r^{(1)}
=
y-\hat{y}^{(0)}
=
[-5,\ -3,\ -1,\ 1,\ 3,\ 5]
$$

平方误差下：

$$
L(y,\hat{y})
=
\frac{1}{2}
(y-\hat{y})^2
$$

对预测值 $\hat{y}$ 求一阶导数：

$$
g
=
\frac{\partial L}{\partial \hat{y}}
=
\hat{y}-y
$$

所以：

$$
g
=
-r
$$

也就是：

$$
g=[5,\ 3,\ 1,\ -1,\ -3,\ -5]
$$

对预测值 $\hat{y}$ 求二阶导数：

$$
h
=
\frac{\partial^2 L}{\partial \hat{y}^2}
=
1
$$

所以：

$$
h=[1,\ 1,\ 1,\ 1,\ 1,\ 1]
$$

说人话：

普通 GBDT 在平方误差下直接拟合残差 $r$。

XGBoost 在平方误差下看的是梯度 $g$，但因为：

$$
g=-r
$$

所以它本质上仍然是在利用残差信息，只是写成了“优化目标函数”的形式。

现在设：

$$
\lambda=1
$$

$$
\gamma=0
$$

我们来比较每个候选切分点。

父节点包含全部样本：

$$
G=5+3+1-1-3-5=0
$$

$$
H=1+1+1+1+1+1=6
$$

候选切分点来自相邻 $x$ 的中点：

$$
1.5,\ 2.5,\ 3.5,\ 4.5,\ 5.5
$$

切分增益公式是：

$$
Gain
=
\frac{1}{2}
\left[
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
-
\frac{G^2}{H+\lambda}
\right]
-
\gamma
$$

因为这里：

$$
G=0
$$

并且：

$$
\gamma=0
$$

所以公式变成：

$$
Gain
=
\frac{1}{2}
\left[
\frac{G_L^2}{H_L+1}
+
\frac{G_R^2}{H_R+1}
\right]
$$

### 切分点 $s=1.5$

左边样本：

$$
x\le 1.5
$$

左边只有第 $1$ 个样本：

$$
G_L=5
$$

$$
H_L=1
$$

右边是其余 $5$ 个样本：

$$
G_R=3+1-1-3-5=-5
$$

$$
H_R=5
$$

所以：

$$
Gain
=
\frac{1}{2}
\left[
\frac{5^2}{1+1}
+
\frac{(-5)^2}{5+1}
\right]
$$

$$
Gain
=
\frac{1}{2}
\left[
12.5+4.1667
\right]
$$

$$
Gain
=
8.3333
$$

### 切分点 $s=2.5$

左边样本：

$$
x\le 2.5
$$

$$
G_L=5+3=8
$$

$$
H_L=2
$$

右边样本：

$$
G_R=1-1-3-5=-8
$$

$$
H_R=4
$$

所以：

$$
Gain
=
\frac{1}{2}
\left[
\frac{8^2}{2+1}
+
\frac{(-8)^2}{4+1}
\right]
$$

$$
Gain
=
\frac{1}{2}
\left[
21.3333+12.8
\right]
$$

$$
Gain
=
17.0667
$$

### 切分点 $s=3.5$

左边样本：

$$
x\le 3.5
$$

$$
G_L=5+3+1=9
$$

$$
H_L=3
$$

右边样本：

$$
G_R=-1-3-5=-9
$$

$$
H_R=3
$$

所以：

$$
Gain
=
\frac{1}{2}
\left[
\frac{9^2}{3+1}
+
\frac{(-9)^2}{3+1}
\right]
$$

$$
Gain
=
\frac{1}{2}
\left[
20.25+20.25
\right]
$$

$$
Gain
=
20.25
$$

### 切分点 $s=4.5$

$$
G_L=5+3+1-1=8
$$

$$
H_L=4
$$

$$
G_R=-3-5=-8
$$

$$
H_R=2
$$

所以：

$$
Gain
=
\frac{1}{2}
\left[
\frac{8^2}{4+1}
+
\frac{(-8)^2}{2+1}
\right]
$$

$$
Gain
=
17.0667
$$

### 切分点 $s=5.5$

$$
G_L=5+3+1-1-3=5
$$

$$
H_L=5
$$

$$
G_R=-5
$$

$$
H_R=1
$$

所以：

$$
Gain
=
\frac{1}{2}
\left[
\frac{5^2}{5+1}
+
\frac{(-5)^2}{1+1}
\right]
$$

$$
Gain
=
8.3333
$$

汇总：

| 切分点 | $G_L$ | $H_L$ | $G_R$ | $H_R$ | $Gain$ |
|---:|---:|---:|---:|---:|---:|
| $1.5$ | $5$ | $1$ | $-5$ | $5$ | $8.3333$ |
| $2.5$ | $8$ | $2$ | $-8$ | $4$ | $17.0667$ |
| $3.5$ | $9$ | $3$ | $-9$ | $3$ | $20.25$ |
| $4.5$ | $8$ | $4$ | $-8$ | $2$ | $17.0667$ |
| $5.5$ | $5$ | $5$ | $-5$ | $1$ | $8.3333$ |

最大的是：

$$
s=3.5
$$

所以第一棵树选择：

$$
x\le 3.5
$$

作为切分规则。

接下来算左右叶子的输出。

左叶子：

$$
w_L
=
-
\frac{G_L}{H_L+\lambda}
=
-
\frac{9}{3+1}
=
-2.25
$$

右叶子：

$$
w_R
=
-
\frac{G_R}{H_R+\lambda}
=
-
\frac{-9}{3+1}
=
2.25
$$

所以这棵树的输出是：

$$
f_1(x)
=
\begin{cases}
-2.25,& x\le 3.5\\
2.25,& x>3.5
\end{cases}
$$

对比普通 GBDT：

普通 GBDT 在这个例子里左右叶子输出是残差均值：

$$
-3,\quad 3
$$

XGBoost 因为有 $\lambda=1$，把叶子输出压小了：

$$
-2.25,\quad 2.25
$$

说人话：

黑马程序员那种 GBDT 手算，一般是在比较“拟合残差后的平方误差”。

XGBoost 这里是在比较“目标函数降低了多少”。

在平方误差场景下，两者方向是一致的：都是想找一个能最好解释当前残差的切分。

但是 XGBoost 多考虑了：

- 二阶梯度 $h$。
- 叶子输出惩罚 $\lambda$。
- 新增叶子的复杂度惩罚 $\gamma$。

所以它的计算表面上和普通 GBDT 不一样，但底层目标仍然是：让这一轮新树把旧模型没解释好的部分补上。

In [ ]:
import numpy as np
import pandas as pd

y = np.array([2, 4, 6, 8, 10, 12], dtype=float)
prediction = np.ones_like(y) * y.mean()
residual = y - prediction
g = prediction - y
h = np.ones_like(y)

def split_gain(g, h, left_mask, lambda_=1.0, gamma=0.0):
    right_mask = ~left_mask
    G_L = g[left_mask].sum()
    H_L = h[left_mask].sum()
    G_R = g[right_mask].sum()
    H_R = h[right_mask].sum()
    G = G_L + G_R
    H = H_L + H_R
    gain = 0.5 * (
        G_L ** 2 / (H_L + lambda_)
        + G_R ** 2 / (H_R + lambda_)
        - G ** 2 / (H + lambda_)
    ) - gamma
    w_left = -G_L / (H_L + lambda_)
    w_right = -G_R / (H_R + lambda_)
    return gain, w_left, w_right

x = np.array([1, 2, 3, 4, 5, 6], dtype=float)
thresholds = [1.5, 2.5, 3.5, 4.5, 5.5]

rows = []
for threshold in thresholds:
    left_mask = x <= threshold
    gain, w_left, w_right = split_gain(g, h, left_mask, lambda_=1.0, gamma=0.0)
    rows.append({
        "切分点": threshold,
        "左叶子输出": w_left,
        "右叶子输出": w_right,
        "切分增益": gain,
    })

pd.DataFrame(rows).sort_values("切分增益", ascending=False)

## 11. XGBoost 的学习率

XGBoost 也有学习率，通常写成：

$$
\eta
$$

在代码参数里经常叫：

$$
learning\_rate
$$

注意：树本身先算出叶子输出：

$$
w_j
$$

真正加入模型时，会乘上学习率：

$$
F_t(x)
=
F_{t-1}(x)
+
\eta f_t(x)
$$

如果刚才第一棵树的叶子输出是：

$$
f_1(x)
=
\begin{cases}
-2.25,& x\le 3.5\\
2.25,& x>3.5
\end{cases}
$$

并且学习率取：

$$
\eta=0.5
$$

那么真正加到模型里的不是：

$$
-2.25,\quad 2.25
$$

而是：

$$
-1.125,\quad 1.125
$$

所以更新后的预测是：

$$
F_1(x)
=
F_0(x)
+
0.5f_1(x)
$$

对左边三个样本：

$$
F_1(x)
=
7-1.125
=
5.875
$$

对右边三个样本：

$$
F_1(x)
=
7+1.125
=
8.125
$$

说人话：

学习率不是改变“这棵树怎么被训练出来”的核心公式。

它改变的是：这棵树训练好以后，往总模型里加多少。

$\eta$ 越小，每一棵树说话越轻，模型学得越慢，也通常需要更多树。

## 12. 本地环境说明

当前本地环境还没有安装 `xgboost` 包。

下面的代码只检查是否安装，不会报错中断 notebook。

In [ ]:
try:
    import xgboost as xgb
    result = f"xgboost 已安装，版本：{xgb.__version__}"
except Exception as e:
    result = f"当前环境未安装 xgboost：{type(e).__name__}"

result

## 13. 本课小结

XGBoost 相比普通 GBDT，关键增强有三点：

第一，目标函数里加入正则化，也就是同时考虑训练误差和树复杂度惩罚。

第二，使用一阶梯度和二阶梯度：

$$
g_i,\quad h_i
$$

第三，用切分增益决定是否切分：

$$
Gain>0
$$

才值得切。

和 GBDT 的关系：

普通 GBDT 在平方误差下，叶子输出是残差均值。

XGBoost 在平方误差下，叶子输出可以看成：

带正则化的残差均值。

一句话：

XGBoost 仍然是 Boosting，但它更精细地衡量每一步怎么长树、叶子输出多大、要不要继续切分。